In [1]:
from pathlib import Path
import pandas as pd

In [7]:
def build_city_hourly_demand(input_file: str) -> pd.DataFrame:
    df = pd.read_csv("D:/projects/chicago-bike-demand-intelligence/02_data_processed/cleaned_divvy_trips.csv")

    df["started_at"] = pd.to_datetime(df["started_at"])

    df["date"] = df["started_at"].dt.date
    df["hour"] = df["started_at"].dt.hour
    df["day_of_week"] = df["started_at"].dt.dayofweek

    df["is_weekend"] = df["day_of_week"].isin([5, 6])
    df["rush_hour"] = df["hour"].isin([7, 8, 9, 16, 17, 18])

    city_hourly_demand = (
        df.groupby(["date", "hour"], as_index=False)
        .agg(
            trip_count=("ride_id", "count"),
            member_trip_count=("member_casual", lambda x: (x == "member").sum()),
            casual_trip_count=("member_casual", lambda x: (x == "casual").sum()),
            avg_trip_duration_min=("trip_duration_min", "mean"),
            is_weekend=("is_weekend", "max"),
            rush_hour=("rush_hour", "max"),
        )
    )

    city_hourly_demand["avg_trip_duration_min"] = city_hourly_demand["avg_trip_duration_min"].round(2)

    return city_hourly_demand


if __name__ == "__main__":
    input_path = "D:/Projects/chicago-bike-demand-intelligence/03_python/03_transform/cleaned_divvy_trips.csv"
    output_path = Path("D:/Projects/chicago-bike-demand-intelligence/02_data_processed/city_hourly_demand.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    city_hourly_demand = build_city_hourly_demand(input_path)
    city_hourly_demand.to_csv(output_path, index=False)

    print("City hourly demand created")
    print(city_hourly_demand.head())
    print(city_hourly_demand.shape)

City hourly demand created
         date  hour  trip_count  member_trip_count  casual_trip_count  \
0  2025-02-28    22           1                  1                  0   
1  2025-02-28    23          25                 19                  6   
2  2025-03-01     0         122                 69                 53   
3  2025-03-01     1          85                 52                 33   
4  2025-03-01     2          46                 26                 20   

   avg_trip_duration_min  is_weekend  rush_hour  
0                 109.84       False      False  
1                  18.15       False      False  
2                   8.53        True      False  
3                   8.42        True      False  
4                   7.60        True      False  
(2209, 8)


In [5]:
print(city_hourly_demand.isna().sum())

print(
    (city_hourly_demand["trip_count"] ==
     city_hourly_demand["member_trip_count"] + city_hourly_demand["casual_trip_count"]).all()
)

date                     0
hour                     0
trip_count               0
member_trip_count        0
casual_trip_count        0
avg_trip_duration_min    0
is_weekend               0
rush_hour                0
dtype: int64
True
